In [1]:
import geopandas as gpd

import pandas as pd

In [2]:
# config
pd.set_option('display.max_colwidth', None)

In [7]:
shapefile_path = "data/historic_airphoto_index/historic_airphoto_index.shp"

gdf = gpd.read_file(shapefile_path)

print(gdf.crs)

EPSG:3857


In [9]:
# convert to UTM 15
gdf = gdf.to_crs(epsg=26915)

print(gdf.crs)

EPSG:26915


In [10]:
#save a UTM copy of that shapefile just for kicks
gdf.to_file("data/historic_airphoto_index/historic_airphoto_index_utm.shp")

In [ ]:
#preview the headers and some of the data

gdf.columns

gdf

### Preparing the main tracker table

columns to add: 

- download_filename
- output_filename

- geometry_utm (GeoSeries in EPSG:26915)
- ml_confidence
- gcp_count
- error_log

### filenames

In [12]:
# create a safe file name
gdf["download_filename"] = (
    gdf["filename"]
    .astype(str)
    .str.lstrip("/")
    .str.replace("/", "_")
)

In [13]:
# create a safe file name
gdf["output_filename"] = (
    gdf["download_filename"].apply(lambda f: f.replace(".jpg", "_georef.tif"))
)

In [ ]:
gdf

### row sorting

In [51]:
gdf_sorted = gdf.sort_values(by=['project','year'], ascending=[True, True])

gdf_sorted

,OBJECTID_1,filename,provider,project,year,photo_id,ext,xutm,yutm,status,image_url,globalid,geometry,download_filename,output_filename
25182,154847,/projects/ait/y1975/ait_7_162.jpg,projects,ait,1975,ait_7_162,jpg,470070.0,5108393.0,1,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_162.jpg,{CE373997-6112-49D2-9102-72273DF606B2},POINT (470069.123 5108393.671),projects_ait_y1975_ait_7_162.jpg,projects_ait_y1975_ait_7_162_georef.tif
25183,154848,/projects/ait/y1975/ait_7_163.jpg,projects,ait,1975,ait_7_163,jpg,470166.0,5109793.0,1,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_163.jpg,{4B76F451-19FF-4185-BA18-7DF2B1FEE4FE},POINT (470165.516 5109793.468),projects_ait_y1975_ait_7_163.jpg,projects_ait_y1975_ait_7_163_georef.tif
25184,154849,/projects/ait/y1975/ait_7_164.jpg,projects,ait,1975,ait_7_164,jpg,470149.0,5111109.0,1,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_164.jpg,{0C3B9C34-ABE3-4E53-8D13-F2C86CBA6487},POINT (470148.752 5111109.445),projects_ait_y1975_ait_7_164.jpg,projects_ait_y1975_ait_7_164_georef.tif
25185,154850,/projects/ait/y1975/ait_7_165.jpg,projects,ait,1975,ait_7_165,jpg,470221.0,5112290.0,1,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_165.jpg,{86F4FF7A-77BD-4C2D-B925-93CDBEC4F5B0},POINT (470219.999 5112291.309),projects_ait_y1975_ait_7_165.jpg,projects_ait_y1975_ait_7_165_georef.tif
25186,154851,/projects/ait/y1975/ait_7_166.jpg,projects,ait,1975,ait_7_166,jpg,470267.0,5113456.0,1,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_166.jpg,{D0E3C8A8-0D65-4B99-B83E-F08008B6764F},POINT (470266.1 5113456.409),projects_ait_y1975_ait_7_166.jpg,projects_ait_y1975_ait_7_166_georef.tif
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16085,165097,/projects/wp/y1963/wp_3dd_120.jpg,projects,wp,1963,wp_3dd_120,jpg,406051.0,5005517.0,1,https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_120.jpg,{ED680899-C440-443F-9EA8-5EE22C99FF9E},POINT (406050.215 5005518.257),projects_wp_y1963_wp_3dd_120.jpg,projects_wp_y1963_wp_3dd_120_georef.tif
16086,165127,/projects/wp/y1963/wp_3dd_158.jpg,projects,wp,1963,wp_3dd_158,jpg,401160.0,5018139.0,1,https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_158.jpg,{562E7926-0A26-4A6C-8215-40C0E844C537},POINT (401159.582 5018139.933),projects_wp_y1963_wp_3dd_158.jpg,projects_wp_y1963_wp_3dd_158_georef.tif
16087,165128,/projects/wp/y1963/wp_3dd_159.jpg,projects,wp,1963,wp_3dd_159,jpg,402869.0,5018054.0,1,https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_159.jpg,{8651A8B0-984A-4553-A1A1-E370031940AB},POINT (402868.1 5018054.507),projects_wp_y1963_wp_3dd_159.jpg,projects_wp_y1963_wp_3dd_159_georef.tif
16088,165129,/projects/wp/y1963/wp_3dd_160.jpg,projects,wp,1963,wp_3dd_160,jpg,404150.0,5018096.0,1,https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_160.jpg,{C7CBE4EF-139A-4190-9532-D3EE8A9F8C45},POINT (404149.488 5018097.22),projects_wp_y1963_wp_3dd_160.jpg,projects_wp_y1963_wp_3dd_160_georef.tif


In [52]:
# save that

gdf = gdf_sorted

### column sorting

In [55]:
# inspect
gdf.columns

Index(['OBJECTID_1', 'filename', 'provider', 'project', 'year', 'photo_id',
       'ext', 'xutm', 'yutm', 'status', 'image_url', 'globalid', 'geometry',
       'download_filename', 'output_filename'],
      dtype='str')

In [57]:
gdf_columnorder = gdf[['OBJECTID_1', 'filename', 'project', 'year', 'provider', 'geometry', 'image_url', 
                       'download_filename', 'output_filename', 'photo_id', 'status', 'xutm', 'yutm', 'ext', 'globalid']]

gdf_columnorder

,OBJECTID_1,filename,project,year,provider,geometry,image_url,download_filename,output_filename,photo_id,status,xutm,yutm,ext,globalid
25182,154847,/projects/ait/y1975/ait_7_162.jpg,ait,1975,projects,POINT (470069.123 5108393.671),https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_162.jpg,projects_ait_y1975_ait_7_162.jpg,projects_ait_y1975_ait_7_162_georef.tif,ait_7_162,1,470070.0,5108393.0,jpg,{CE373997-6112-49D2-9102-72273DF606B2}
25183,154848,/projects/ait/y1975/ait_7_163.jpg,ait,1975,projects,POINT (470165.516 5109793.468),https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_163.jpg,projects_ait_y1975_ait_7_163.jpg,projects_ait_y1975_ait_7_163_georef.tif,ait_7_163,1,470166.0,5109793.0,jpg,{4B76F451-19FF-4185-BA18-7DF2B1FEE4FE}
25184,154849,/projects/ait/y1975/ait_7_164.jpg,ait,1975,projects,POINT (470148.752 5111109.445),https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_164.jpg,projects_ait_y1975_ait_7_164.jpg,projects_ait_y1975_ait_7_164_georef.tif,ait_7_164,1,470149.0,5111109.0,jpg,{0C3B9C34-ABE3-4E53-8D13-F2C86CBA6487}
25185,154850,/projects/ait/y1975/ait_7_165.jpg,ait,1975,projects,POINT (470219.999 5112291.309),https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_165.jpg,projects_ait_y1975_ait_7_165.jpg,projects_ait_y1975_ait_7_165_georef.tif,ait_7_165,1,470221.0,5112290.0,jpg,{86F4FF7A-77BD-4C2D-B925-93CDBEC4F5B0}
25186,154851,/projects/ait/y1975/ait_7_166.jpg,ait,1975,projects,POINT (470266.1 5113456.409),https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_166.jpg,projects_ait_y1975_ait_7_166.jpg,projects_ait_y1975_ait_7_166_georef.tif,ait_7_166,1,470267.0,5113456.0,jpg,{D0E3C8A8-0D65-4B99-B83E-F08008B6764F}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16085,165097,/projects/wp/y1963/wp_3dd_120.jpg,wp,1963,projects,POINT (406050.215 5005518.257),https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_120.jpg,projects_wp_y1963_wp_3dd_120.jpg,projects_wp_y1963_wp_3dd_120_georef.tif,wp_3dd_120,1,406051.0,5005517.0,jpg,{ED680899-C440-443F-9EA8-5EE22C99FF9E}
16086,165127,/projects/wp/y1963/wp_3dd_158.jpg,wp,1963,projects,POINT (401159.582 5018139.933),https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_158.jpg,projects_wp_y1963_wp_3dd_158.jpg,projects_wp_y1963_wp_3dd_158_georef.tif,wp_3dd_158,1,401160.0,5018139.0,jpg,{562E7926-0A26-4A6C-8215-40C0E844C537}
16087,165128,/projects/wp/y1963/wp_3dd_159.jpg,wp,1963,projects,POINT (402868.1 5018054.507),https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_159.jpg,projects_wp_y1963_wp_3dd_159.jpg,projects_wp_y1963_wp_3dd_159_georef.tif,wp_3dd_159,1,402869.0,5018054.0,jpg,{8651A8B0-984A-4553-A1A1-E370031940AB}
16088,165129,/projects/wp/y1963/wp_3dd_160.jpg,wp,1963,projects,POINT (404149.488 5018097.22),https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_160.jpg,projects_wp_y1963_wp_3dd_160.jpg,projects_wp_y1963_wp_3dd_160_georef.tif,wp_3dd_160,1,404150.0,5018096.0,jpg,{C7CBE4EF-139A-4190-9532-D3EE8A9F8C45}


## Calculating extents from center points

In match_multi_zoom, the plugin ultimately calls:
tile_fetcher.fetch_tiles_from_center(
    center_lat,
    center_lon,
    extent_meters,
    extent_dimension="horizontal",
    source_aspect_ratio,
    basemap_source,
    zoom_level
)

The function definition tells us exactly what extent_meters represents:
extent_meters = width (horizontal) OR height (vertical) of the basemap area, in meters [mn365-my.s...epoint.com]
…which is then converted into a rectangle in EPSG:3857:
half_width = width_meters / 2
half_height = height_meters / 2

extent = QgsRectangle(
    center_3857.x() - half_width,
    center_3857.y() - half_height,
    center_3857.x() + half_width,
    center_3857.y() + half_height
)

So the extent_meters must be a real-world width (or height) of your historic air photo footprint.

We just need 1 dimension - probably the height? 

We can get the dimensions and DPI of the photo from the JPEG, but in order to know its real world size, we'll need its scale. 

### How much variation in scale is there due to different flight series? 

https://www.dnr.state.mn.us/maps/historical_airphotos/index.html

How feasible is it for me to guess at the scale? How would I divide this into chunks, if needed? 

In [22]:
# how many different categories are these photos in? 
provider_values = gdf['provider'].nunique()
print(provider_values)

provider_counts = gdf['provider'].value_counts(dropna=False)
print(provider_counts)

print("---------")

project_values = gdf['project'].nunique()
print(project_values)

project_counts = gdf['project'].value_counts(dropna=False)
print(project_counts)

print("---------")
year_values = gdf['year'].nunique()
print(year_values)

year_counts = gdf['year'].value_counts(dropna=False)
print(year_counts)

3
provider
projects    48034
usda        43603
usgs         2124
Name: count, dtype: int64
---------
106
project
stl     9977
slc     8776
czm     8037
cir     7127
koo     4493
        ... 
wn        26
cas       24
eui       24
ca        12
cass       9
Name: count, Length: 106, dtype: int64
---------
44
year
1939    8947
1989    7236
1938    7080
1972    6421
1961    6316
1981    5068
1948    4522
1955    4222
1954    3882
1951    3431
2012    2995
2007    2722
1947    2430
1988    2340
2002    2320
1953    2312
1982    2183
1980    2153
1963    2051
1976    1932
1965    1865
1940    1763
1949    1406
1966    1378
1969    1148
1975    1018
1959     667
1978     442
1957     426
1991     401
1964     361
1973     358
1934     347
1950     339
1960     290
1962     217
1958     143
1941     135
1936     132
1937     130
1952     100
1956      54
1968      54
1979      24
Name: count, dtype: int64


In [27]:
gdf

,OBJECTID_1,filename,provider,project,year,photo_id,ext,xutm,yutm,status,image_url,globalid,geometry,download_filename,output_filename
0,17660,/usda/bjf/y1938/bjf02002.jpg,usda,bjf,1938,bjf02002,jpg,260432.0,4868321.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02002.jpg,{6C21141B-1856-46E2-B199-FB9678486151},POINT (260431.708 4868321.904),usda_bjf_y1938_bjf02002.jpg,usda_bjf_y1938_bjf02002_georef.tif
1,17662,/usda/bjf/y1938/bjf02004.jpg,usda,bjf,1938,bjf02004,jpg,257333.0,4868367.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02004.jpg,{967F845D-5444-40B6-BF05-91437D1CAF63},POINT (257332.011 4868368.265),usda_bjf_y1938_bjf02004.jpg,usda_bjf_y1938_bjf02004_georef.tif
2,17664,/usda/bjf/y1938/bjf02006.jpg,usda,bjf,1938,bjf02006,jpg,254233.0,4868414.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02006.jpg,{04683AF5-AE59-4778-AC4B-849119DEC0B0},POINT (254232.313 4868414.627),usda_bjf_y1938_bjf02006.jpg,usda_bjf_y1938_bjf02006_georef.tif
3,17665,/usda/bjf/y1938/bjf02012.jpg,usda,bjf,1938,bjf02012,jpg,253965.0,4865651.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02012.jpg,{8245AC47-1BED-42D5-81E2-79CDEF7E44BF},POINT (253964.528 4865651.395),usda_bjf_y1938_bjf02012.jpg,usda_bjf_y1938_bjf02012_georef.tif
4,17667,/usda/bjf/y1938/bjf02014.jpg,usda,bjf,1938,bjf02014,jpg,256931.0,4865569.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02014.jpg,{4F0BDE31-A7AA-4781-A669-29CF8E7C3DE2},POINT (256930.424 4865569.669),usda_bjf_y1938_bjf02014.jpg,usda_bjf_y1938_bjf02014_georef.tif
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93756,96459,/usda/byg/y1966/byg05gg117.jpg,usda,byg,1966,byg05gg117,jpg,319917.0,5429225.0,1,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg117.jpg,{FFE59653-BF0D-4BFE-AA97-F5B512DEAA4B},POINT (319916.86 5429226.027),usda_byg_y1966_byg05gg117.jpg,usda_byg_y1966_byg05gg117_georef.tif
93757,96460,/usda/byg/y1966/byg05gg119.jpg,usda,byg,1966,byg05gg119,jpg,323196.0,5429165.0,1,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg119.jpg,{FB5FE609-92DD-4B07-AC46-624C3DB7959D},POINT (323195.16 5429166.422),usda_byg_y1966_byg05gg119.jpg,usda_byg_y1966_byg05gg119_georef.tif
93758,96461,/usda/byg/y1966/byg05gg121.jpg,usda,byg,1966,byg05gg121,jpg,326395.0,5429046.0,1,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg121.jpg,{2933DB1E-682F-464A-99F8-369190014377},POINT (326393.986 5429047.211),usda_byg_y1966_byg05gg121.jpg,usda_byg_y1966_byg05gg121_georef.tif
93759,96462,/usda/byg/y1966/byg05gg123.jpg,usda,byg,1966,byg05gg123,jpg,329683.0,5428977.0,1,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg123.jpg,{7E07CF60-6D46-4F27-8107-A47602220FB5},POINT (329682.221 5428977.671),usda_byg_y1966_byg05gg123.jpg,usda_byg_y1966_byg05gg123_georef.tif


In [44]:
filtered = gdf.loc[(gdf['year']==1989) & (gdf['project']=='stl')]

filtered

,OBJECTID_1,filename,provider,project,year,photo_id,ext,xutm,yutm,status,image_url,globalid,geometry,download_filename,output_filename
27152,137998,/projects/stl/y1989/stl_029_120.jpg,projects,stl,1989,stl_029_120,jpg,561926.0,5172458.0,1,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_029_120.jpg,{CCB4EDBA-7BEA-425C-9303-5A2995B7E585},POINT (561925.715 5172458.418),projects_stl_y1989_stl_029_120.jpg,projects_stl_y1989_stl_029_120_georef.tif
27153,137999,/projects/stl/y1989/stl_029_121.jpg,projects,stl,1989,stl_029_121,jpg,562024.0,5170972.0,1,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_029_121.jpg,{0E3E4AC3-9EDB-42A8-A2BE-DC5A57EF0DF6},POINT (562023.028 5170972.418),projects_stl_y1989_stl_029_121.jpg,projects_stl_y1989_stl_029_121_georef.tif
27154,138000,/projects/stl/y1989/stl_029_122.jpg,projects,stl,1989,stl_029_122,jpg,562037.0,5169572.0,1,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_029_122.jpg,{DE530966-CCE1-4A96-BBAC-C6538B5439B2},POINT (562036.84 5169573.418),projects_stl_y1989_stl_029_122.jpg,projects_stl_y1989_stl_029_122_georef.tif
27155,138001,/projects/stl/y1989/stl_029_123.jpg,projects,stl,1989,stl_029_123,jpg,562118.0,5168054.0,1,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_029_123.jpg,{2BBFC083-E4FB-4082-B619-FF6BBD12FD0C},POINT (562117.278 5168054.917),projects_stl_y1989_stl_029_123.jpg,projects_stl_y1989_stl_029_123_georef.tif
27156,138002,/projects/stl/y1989/stl_029_124.jpg,projects,stl,1989,stl_029_124,jpg,562127.0,5166598.0,1,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_029_124.jpg,{A9946C10-16AE-4251-8580-E4FA89AB6AA9},POINT (562126.715 5166598.417),projects_stl_y1989_stl_029_124.jpg,projects_stl_y1989_stl_029_124_georef.tif
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89123,142855,/projects/stl/y1989/stl_003_001.jpg,projects,stl,1989,stl_003_001,jpg,497585.0,5367304.0,1,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_003_001.jpg,{E974D5A4-DB61-4DB4-ADA1-3A9EF32A52B7},POINT (497583.986 5367304.444),projects_stl_y1989_stl_003_001.jpg,projects_stl_y1989_stl_003_001_georef.tif
89124,142863,/projects/stl/y1989/stl_001_001.jpg,projects,stl,1989,stl_001_001,jpg,492724.0,5373348.0,1,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_001_001.jpg,{54B0EAE2-299F-4D8A-9C6B-E973F1A662CD},POINT (492722.954 5373348.445),projects_stl_y1989_stl_001_001.jpg,projects_stl_y1989_stl_001_001_georef.tif
89125,142864,/projects/stl/y1989/stl_001_002.jpg,projects,stl,1989,stl_001_002,jpg,492744.0,5372073.0,1,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_001_002.jpg,{12B1A60D-9A14-4AE0-9ADA-D5D330070D2E},POINT (492743.329 5372073.945),projects_stl_y1989_stl_001_002.jpg,projects_stl_y1989_stl_001_002_georef.tif
89126,142865,/projects/stl/y1989/stl_001_003.jpg,projects,stl,1989,stl_001_003,jpg,492696.0,5370790.0,1,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_001_003.jpg,{AC6147AA-F096-4F31-BBA9-D1D46F61C80C},POINT (492695.829 5370791.444),projects_stl_y1989_stl_001_003.jpg,projects_stl_y1989_stl_001_003_georef.tif


In [ ]:
project_counts = filtered['project'].value_counts(dropna=False)
print(project_counts)

print("-------")

provider_counts = filtered['provider'].value_counts(dropna=False)
print(provider_counts)

### Approach to calculating extents

pixel_size_on_film_m  = 0.0254 / dpi
ground_pixel_size_m   = pixel_size_on_film_m * scale
extent_width_meters   = width_pixels * ground_pixel_size_m
extent_height_meters  = height_pixels * ground_pixel_size_m

Then send to magic georef: 
extent_meters = extent_width_meters  # Magic Georeferencer prefers horizontal
extent_dimension = "horizontal"

#### Next steps

1. Choose a working subset. I think I want to start with 3 photos each from 10 different years, all in St. Louis county. Save this in memory as gdf_test1
2. 